In [2]:
!pip install -q -U kaggle
!pip install -q holidays prophet ortools dash darts

import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

for d in ["data", "outputs", "screenshots"]:
    os.makedirs(f"/content/DS-02/{d}", exist_ok=True)
%cd /content/DS-02
!kaggle competitions download -c demand-forecasting-kernels-only -p data
!unzip -o -q data/demand-forecasting-kernels-only.zip -d data
!ls data

/content/DS-02
100% 3.29M/3.29M [00:00<00:00, 38.7MB/s]

demand-forecasting-kernels-only.zip  sample_submission.csv  test.csv  train.csv


In [3]:
import pandas as pd, numpy as np, holidays
df = pd.read_csv("data/train.csv", parse_dates=["date"]).sort_values(["store","item","date"])
g = df.groupby(["store","item"])["sales"]

for lag in [7, 14, 28]:
    df[f"lag_{lag}"] = g.shift(lag)
df["roll_mean_7"]  = g.transform(lambda s: s.shift(7).rolling(7).mean())
df["roll_mean_28"] = g.transform(lambda s: s.shift(7).rolling(28).mean())

df["dow"] = df.date.dt.dayofweek
df["month"] = df.date.dt.month
us_h = pd.to_datetime(list(holidays.US(years=range(2013, 2019)).keys()))
df["is_holiday"] = df.date.isin(us_h).astype(int)
df = df.dropna().reset_index(drop=True)
print(df.shape)

(896000, 12)


In [4]:
import lightgbm as lgb, json
cut = df.date.max() - pd.Timedelta(days=90)
train, test = df[df.date <= cut].copy(), df[df.date > cut].copy()
feats = ["store","item","lag_7","lag_14","lag_28","roll_mean_7","roll_mean_28","dow","month","is_holiday"]

def wape(y, p): return float(np.abs(y - p).sum() / np.abs(y).sum())
def rmsse(train, test, col):
    scale = train.groupby(["store","item"])["sales"].apply(lambda s: np.mean(np.diff(s.values) ** 2))
    err = ((test["sales"] - test[col]) ** 2).groupby([test["store"], test["item"]]).mean()
    return float(np.mean(np.sqrt(err / scale)))

test["naive"] = test["lag_7"]
m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1).fit(train[feats], train["sales"])
test["pred"] = m.predict(test[feats])
for a, name in [(0.1, "p10"), (0.9, "p90")]:
    q = lgb.LGBMRegressor(objective="quantile", alpha=a, n_estimators=300, verbose=-1).fit(train[feats], train["sales"])
    test[name] = q.predict(test[feats])

metrics = {
    "naive_wape": wape(test.sales, test.naive), "lgbm_wape": wape(test.sales, test.pred),
    "naive_rmsse": rmsse(train, test, "naive"), "lgbm_rmsse": rmsse(train, test, "pred"),
    "interval_coverage_80": float(((test.sales >= test.p10) & (test.sales <= test.p90)).mean()),
}
print(metrics)
json.dump(metrics, open("outputs/metrics.json", "w"), indent=2)
test[["date","store","item","sales","naive","pred","p10","p90"]].to_csv("outputs/test_predictions.csv", index=False)

{'naive_wape': 0.16600085665159867, 'lgbm_wape': 0.11318945347661898, 'naive_rmsse': 0.8723418847494754, 'lgbm_rmsse': 0.5890716608211233, 'interval_coverage_80': 0.785}


In [5]:
from prophet import Prophet
raw = pd.read_csv("data/train.csv", parse_dates=["date"])
s = raw[(raw.store == 1) & (raw.item == 1)][["date","sales"]].rename(columns={"date":"ds","sales":"y"})
tr, te = s.iloc[:-90], s.iloc[-90:]
fc = Prophet(yearly_seasonality=True, weekly_seasonality=True).fit(tr).predict(te[["ds"]])
pw = float(np.abs(te.y.values - fc.yhat.values).sum() / te.y.sum())
print("Prophet WAPE (store 1, item 1):", pw)

# Same series par LightGBM ka WAPE, fair comparison ke liye
t11 = test[(test.store == 1) & (test.item == 1)]
print("LightGBM WAPE (store 1, item 1):", wape(t11.sales, t11.pred))
print("Naive WAPE (store 1, item 1):", wape(t11.sales, t11.naive))
json.dump({"prophet_wape_store1_item1": pw}, open("outputs/prophet_metrics.json", "w"))

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Prophet WAPE (store 1, item 1): 0.18518891321611558
LightGBM WAPE (store 1, item 1): 0.2019937300146162
Naive WAPE (store 1, item 1): 0.2538293216630197


In [6]:
from darts import TimeSeries
from darts.models import LightGBMModel
from darts.dataprocessing.transformers import MinTReconciliator

raw = pd.read_csv("data/train.csv", parse_dates=["date"])
wide = raw.pivot_table(index="date", columns="store", values="sales", aggfunc="sum")
wide.columns = [f"store_{c}" for c in wide.columns]
wide["total"] = wide.sum(axis=1)
hierarchy = {c: ["total"] for c in wide.columns if c != "total"}
series = TimeSeries.from_dataframe(wide.astype(float), freq="D", hierarchy=hierarchy)

tr_s, val_s = series[:-90], series[-90:]
dm = LightGBMModel(lags=28, output_chunk_length=7, verbose=-1).fit(tr_s)
fc = dm.predict(90)

rec = MinTReconciliator(method="wls_val").fit(tr_s)
fc_rec = rec.transform(fc)

stores = [c for c in wide.columns if c != "total"]
def gap(f):
    d = f.to_dataframe(); return float((d[stores].sum(axis=1) - d["total"]).abs().mean())
def tw(f):
    a, p = val_s.to_dataframe()["total"], f.to_dataframe()["total"]
    return float((a - p).abs().sum() / a.abs().sum())

print("Pehle : gap =", gap(fc), " Total WAPE =", tw(fc))
print("Baad  : gap =", gap(fc_rec), " Total WAPE =", tw(fc_rec))
json.dump({"gap_before": gap(fc), "gap_after": gap(fc_rec)}, open("outputs/reconcile_metrics.json", "w"))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

Pehle : gap = 241.54363953489144  Total WAPE = 0.1129724565663799
Baad  : gap = 2.708273111946053e-12  Total WAPE = 0.112637121538929


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

In [8]:
print("Before : gap =", gap(fc), " Total WAPE =", tw(fc))
print("After : gap =", gap(fc_rec), " Total WAPE =", tw(fc_rec))

Before : gap = 241.54363953489144  Total WAPE = 0.1129724565663799
After : gap = 2.708273111946053e-12  Total WAPE = 0.112637121538929


In [14]:
!pip install -q "u8darts[torch]"
!pip install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 21.0 MB/s  0:00:00


In [15]:
import importlib
for name in ["torch", "pytorch_lightning", "lightning"]:
    try:
        m = importlib.import_module(name)
        print(name, "OK", m.__version__)
    except Exception as e:
        print(name, "MISSING:", repr(e)[:100])

torch OK 2.14.1+cpu
pytorch_lightning OK 2.5.2
lightning OK 2.6.6


In [16]:
from darts.models import TFTModel
print("TFTModel import OK")

TFTModel import OK


In [1]:
import os, json, warnings
warnings.filterwarnings("ignore")
os.chdir("/content/DS-02")

import numpy as np, pandas as pd
from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from darts.utils.likelihood_models import QuantileRegression

raw = pd.read_csv("data/train.csv", parse_dates=["date"])
sub = raw[(raw.store == 1) & (raw.item <= 10)].copy()
sub["sales"] = sub["sales"].astype(float)
ser = TimeSeries.from_group_dataframe(sub, group_cols=["item"], time_col="date", value_cols="sales", freq="D")

H = 30
tr_l = [x[:-H] for x in ser]; va_l = [x[-H:] for x in ser]
scaler = Scaler(); tr_sc = scaler.fit_transform(tr_l)

tft = TFTModel(input_chunk_length=60, output_chunk_length=H, hidden_size=32, lstm_layers=1,
               num_attention_heads=2, dropout=0.1, batch_size=64, n_epochs=15,
               add_relative_index=True, likelihood=QuantileRegression([0.1, 0.5, 0.9]),
               random_state=42, pl_trainer_kwargs={"accelerator": "auto"})
tft.fit(tr_sc, verbose=True)
pr = scaler.inverse_transform(tft.predict(n=H, series=tr_sc, num_samples=200))

a = np.concatenate([v.values().ravel() for v in va_l])
p = np.concatenate([x.quantile(0.5).values().ravel() for x in pr])
tw_ = float(np.abs(a - p).sum() / a.sum())
print("TFT WAPE (store 1, items 1-10, 30 din):", tw_)
json.dump({"tft_wape_subset": tw_}, open("outputs/tft_metrics.json", "w"))

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0      | train
1  | val_metrics                       | MetricCollection                 | 0      | train
2  | input_embeddings                  | _MultiEmbedding                  | 0      | train
3  | static_covariates_vsn             | _VariableSelectionNetwork        | 896    | train
4  | encoder_vsn                       | _VariableSelectionNetwork        | 1.9 K  | train
5  | decoder_vsn                       | _VariableSelectionNetwork   

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=15` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

TFT WAPE (store 1, items 1-10, 30 din): 0.3531121933054345


In [2]:
!ls /content/DS-02/outputs /content/DS-02/data

/content/DS-02/data:
demand-forecasting-kernels-only.zip  sample_submission.csv  test.csv  train.csv

/content/DS-02/outputs:
metrics.json	      reconcile_metrics.json  tft_metrics.json
prophet_metrics.json  test_predictions.csv


In [4]:
import traceback
try:
    import torch, pytorch_lightning, lightning
    from darts.models.forecasting.tft_model import TFTModel
    print("All good")
except Exception:
    traceback.print_exc()

All good


In [5]:
import json, glob
for f in sorted(glob.glob("/content/DS-02/outputs/*.json")):
    print(f.split("/")[-1], json.load(open(f)))

metrics.json {'naive_wape': 0.16600085665159867, 'lgbm_wape': 0.11318945347661898, 'naive_rmsse': 0.8723418847494754, 'lgbm_rmsse': 0.5890716608211233, 'interval_coverage_80': 0.785}
prophet_metrics.json {'prophet_wape_store1_item1': 0.18518891321611558}
reconcile_metrics.json {'gap_before': 241.54363953489144, 'gap_after': 2.708273111946053e-12}
tft_metrics.json {'tft_wape_subset': 0.3531121933054345}


In [6]:
%%writefile /content/DS-02/requirements.txt
dash
plotly
pandas
gunicorn

Writing /content/DS-02/requirements.txt


In [8]:
import os, warnings
warnings.filterwarnings("ignore")
os.chdir("/content/DS-02")
import pandas as pd
from ortools.linear_solver import pywraplp

p1 = pd.read_csv("outputs/test_predictions.csv", parse_dates=["date"])
p1 = p1[p1.store == 1]
first7 = p1[p1.date < p1.date.min() + pd.Timedelta(days=7)]
d = first7.groupby("item")[["p10", "pred", "p90"]].sum().rename(columns={"p10": "low", "pred": "mid", "p90": "high"})

UNIT_COST, HOLD, SHORT = 10.0, 1.0, 4.0
PROB = {"low": 0.25, "mid": 0.5, "high": 0.25}
BUDGET = 0.85 * UNIT_COST * d["high"].sum()

s = pywraplp.Solver.CreateSolver("GLOP")
q = {i: s.NumVar(0, s.infinity(), f"q_{i}") for i in d.index}
obj = []
for i in d.index:
    for sc, pr_ in PROB.items():
        ov, un = s.NumVar(0, s.infinity(), ""), s.NumVar(0, s.infinity(), "")
        s.Add(q[i] - d.loc[i, sc] == ov - un)
        obj.append(pr_ * (HOLD * ov + SHORT * un))
s.Add(sum(UNIT_COST * q[i] for i in d.index) <= BUDGET)
s.Minimize(sum(obj))
assert s.Solve() == pywraplp.Solver.OPTIMAL

out = d.copy()
out["order_qty"] = [round(q[i].solution_value()) for i in d.index]
out.reset_index().to_csv("outputs/orders.csv", index=False)
print(out.head(10)); print("Budget:", round(BUDGET), " Spend:", round((out.order_qty * UNIT_COST).sum()))

             low         mid        high  order_qty
item                                               
1     116.505083  159.865009  203.961748        160
2     348.932560  418.879183  486.331337        419
3     210.062033  265.112331  321.859351        265
4     104.083336  147.260552  186.187996        147
5      92.103240  130.935485  170.888948        131
6     335.890591  405.224224  479.728003        405
7     346.285110  415.441028  487.640877        415
8     482.992504  567.074242  648.890726        567
9     312.983243  380.453899  449.643906        380
10    453.481792  530.123054  614.487635        530
Budget: 187171  Spend: 187160


In [9]:
%%writefile /content/DS-02/app.py
import json
from pathlib import Path
import pandas as pd, plotly.graph_objects as go
from dash import Dash, dcc, html, dash_table, Input, Output

BASE = Path(__file__).parent
pred = pd.read_csv(BASE / "outputs/test_predictions.csv", parse_dates=["date"])
orders = pd.read_csv(BASE / "outputs/orders.csv")
m = json.load(open(BASE / "outputs/metrics.json"))

app = Dash(__name__)
server = app.server
app.layout = html.Div(style={"maxWidth": "1000px", "margin": "auto", "fontFamily": "sans-serif"}, children=[
    html.H2("Demand Forecasting & Inventory Optimization"),
    html.B(f"WAPE LightGBM: {m['lgbm_wape']:.3f} | WAPE Naive: {m['naive_wape']:.3f} | "
           f"RMSSE LightGBM: {m['lgbm_rmsse']:.3f} | 80% interval coverage: {m['interval_coverage_80']:.2f}"),
    html.Br(), html.Br(),
    html.Div([
        html.Label("Store: "),
        dcc.Dropdown(sorted(pred.store.unique()), 1, id="store", clearable=False, style={"width": "120px", "display": "inline-block"}),
        html.Label("  Item: "),
        dcc.Dropdown(sorted(pred.item.unique()), 1, id="item", clearable=False, style={"width": "120px", "display": "inline-block"}),
    ]),
    dcc.Graph(id="chart"),
    html.H3("Order recommendations (Store 1, next 7 days)"),
    dash_table.DataTable(orders.to_dict("records"), page_size=10),
])

@app.callback(Output("chart", "figure"), Input("store", "value"), Input("item", "value"))
def update(store, item):
    d = pred[(pred.store == store) & (pred.item == item)]
    f = go.Figure()
    f.add_scatter(x=d.date, y=d.p90, line=dict(width=0), showlegend=False)
    f.add_scatter(x=d.date, y=d.p10, fill="tonexty", line=dict(width=0), name="80% interval")
    f.add_scatter(x=d.date, y=d.sales, name="Actual")
    f.add_scatter(x=d.date, y=d.pred, name="Forecast")
    f.update_layout(title=f"Store {store}, Item {item}", xaxis_title="Date", yaxis_title="Sales")
    return f

if __name__ == "__main__":
    app.run(debug=False)

Writing /content/DS-02/app.py


In [10]:
import sys; sys.path.insert(0, "/content/DS-02")
import os; os.chdir("/content/DS-02")
from app import app
app.run(jupyter_mode="inline", port=8050)

<IPython.core.display.Javascript object>

In [11]:
%cd /content
!zip -r DS-02.zip DS-02 -x "DS-02/data/*"
from google.colab import files
files.download("DS-02.zip")

/content
  adding: DS-02/ (stored 0%)
  adding: DS-02/outputs/ (stored 0%)
  adding: DS-02/outputs/test_predictions.csv (deflated 57%)
  adding: DS-02/outputs/reconcile_metrics.json (deflated 4%)
  adding: DS-02/outputs/orders.csv (deflated 46%)
  adding: DS-02/outputs/tft_metrics.json (stored 0%)
  adding: DS-02/outputs/metrics.json (deflated 31%)
  adding: DS-02/outputs/prophet_metrics.json (stored 0%)
  adding: DS-02/requirements.txt (stored 0%)
  adding: DS-02/app.py (deflated 53%)
  adding: DS-02/screenshots/ (stored 0%)
  adding: DS-02/__pycache__/ (stored 0%)
  adding: DS-02/__pycache__/app.cpython-313.pyc (deflated 44%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>